# 选修E3 · Day 3 上机：LLM 评估与部署

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

**核心命题**：营销 LLM 上线后，用 deepeval 评估质量、用 langsmith 追踪调用、用 tiktoken 监控成本、用 vLLM/投机解码/MoE 优化推理。

**真实库**：deepeval（评估指标）+ langsmith（追踪）+ tiktoken（token 成本）


## 0. 环境准备

安装并导入真实库：
- **deepeval**：LLM 评估框架，自定义 BaseMetric + LLMTestCase
- **langsmith**：LLM 应用追踪，@traceable 装饰器
- **tiktoken**：token 计数，成本监控基础

In [ ]:
# !pip install deepeval langsmith tiktoken -q

import tiktoken
from dataclasses import dataclass
from typing import List
from deepeval.metrics import BaseMetric
from deepeval.test_case import LLMTestCase
from langsmith import traceable
import time

print("=== 环境就绪 ===")
print(f"tiktoken: {tiktoken.__version__}")
print("deepeval: 可用 (自定义 BaseMetric + LLMTestCase)")
print("langsmith: 可用 (@traceable 本地模式)")


## 1. 场景背景与营销映射

**场景**：智能手表品牌上线了一个营销文案 LLM（自动生成小红书/朋友圈/电商详情页文案）。现在需要：
1. 构建评估集，评估文案质量（准确性/相关性/无害性/忠实性）
2. 追踪部署后每次调用的延迟和 token
3. 监控日均万次请求的推理成本
4. 优化推理架构（vLLM/投机解码/MoE 概念）

**评估集**：5 条真实营销文案样本，覆盖优质/错误/违规/幻觉/跑题五种失败模式。

In [ ]:
# 营销文案评估集（基于真实智能手表产品文档结构构建，见 data/README.md）
@dataclass
class EvalExample:
    input: str              # 输入（产品信息/用户需求）
    actual_output: str      # 待评估的 LLM 生成文案
    expected_output: str    # 参考答案（优质文案）
    criteria: List[str]     # 评估标准
    category: str           # 任务类别

marketing_eval_set = [
    EvalExample(
        input="为智能手表Pro写小红书种草文案，卖点：7天续航、100+运动模式、心率血氧",
        actual_output="姐妹们！这款智能手表Pro真的绝了！续航7天不充电，100+运动模式全覆盖，心率血氧24小时守护。运动女孩必备！快冲！",
        expected_output="告别一天一充的焦虑！智能手表Pro续航7天，100+运动模式，心率血氧实时监测，让你的健康尽在掌握。",
        criteria=["含产品卖点", "符合小红书风格", "有CTA"],
        category="小红书种草",
    ),
    EvalExample(
        input="为智能手表Pro写电商详情页文案",
        actual_output="这款手表续航7天，支持100+运动模式，价格只要99元，还送AirPods。",
        expected_output="智能手表Pro，7天超长续航，100+运动模式，心率血氧监测，5ATM防水。官方售价1299元。",
        criteria=["参数准确", "无虚假宣传", "含价格"],
        category="电商详情页",
    ),
    EvalExample(
        input="为智能手表Pro写朋友圈广告",
        actual_output="这款手表只适合北方人用，南方人别买。",
        expected_output="智能手表Pro，守护全家健康。7天续航，100+运动模式，心率血氧监测。送父母首选。",
        criteria=["无歧视", "无违规", "通用性"],
        category="朋友圈广告",
    ),
    EvalExample(
        input="基于产品知识库写文案，知识库：续航7天、100+运动、5ATM防水",
        actual_output="这款手表续航30天，支持200+运动模式，10ATM防水，还能测血压。",
        expected_output="智能手表Pro续航7天，100+运动模式，5ATM防水，满足日常运动健康需求。",
        criteria=["忠于知识库", "无幻觉", "参数准确"],
        category="RAG生成",
    ),
    EvalExample(
        input="为智能手表Pro写种草文案",
        actual_output="好。",
        expected_output="智能手表Pro，7天续航+100+运动模式+心率血氧，让你的健康尽在掌握。",
        criteria=["含产品卖点", "有CTA", "符合风格"],
        category="小红书种草",
    ),
]

print(f"评测集样本数: {len(marketing_eval_set)}")
for i, ex in enumerate(marketing_eval_set):
    print(f"  [{i+1}] {ex.category}: {ex.actual_output[:35]}...")


## TODO 1：把营销评测集转换为 deepeval LLMTestCase

deepeval 的评估对象是 `LLMTestCase`，需要把上面的 `EvalExample` 转换：
- `input` <- EvalExample.input
- `actual_output` <- EvalExample.actual_output（待评估文案）
- `expected_output` <- EvalExample.expected_output（参考答案）
- `context` <- 提取知识库信息（用于忠实性评估）

**任务**：写 `to_test_cases(eval_set)` 函数，返回 `List[LLMTestCase]`。

In [ ]:
# ===== 你的代码 =====
# 把 EvalExample 转换为 deepeval LLMTestCase 列表
# 提示：LLMTestCase 字段 input / actual_output / expected_output / context
def to_test_cases(eval_set):
    # TODO: 你的代码
    raise NotImplementedError

test_cases = to_test_cases(marketing_eval_set)
print(f"测试用例数: {len(test_cases)}")


## TODO 2：用 deepeval 自定义 MarketingQualityMetric

继承 `BaseMetric`，实现四维度规则评分（无 API，规则近似 LLM-as-a-Judge）：

| 维度 | 规则 | 评分逻辑 |
|------|------|---------|
| **准确性** | actual 与 expected 的关键词重合度 | 重合度高=1.0，中=0.5，低=0.0 |
| **相关性** | actual 是否包含 input 的核心词 | 包含=1.0，部分=0.5，无=0.0 |
| **无害性** | actual 是否含违禁词（歧视/违规） | 无违禁=1.0，有=0.0 |
| **忠实性** | actual 是否忠于 context（无幻觉） | 参数与 context 一致=1.0，编造=0.0 |

**任务**：实现 `measure(test_case)` 和 `is_successful()`。

In [ ]:
# ===== 你的代码 =====
# 继承 deepeval BaseMetric，实现四维度规则评分
class MarketingQualityMetric(BaseMetric):
    def __init__(self, threshold=0.6):
        self.threshold = threshold
        self.score = 0.0
        self.score_breakdown = {}
        self.reason = ""
        self.success = False

    def measure(self, test_case, *args, **kwargs):
        # TODO: 你的代码 —— 实现四维度评分
        # 维度1: 准确性 (actual vs expected 关键词重合)
        # 维度2: 相关性 (actual 是否含 input 核心词)
        # 维度3: 无害性 (是否含违禁词)
        # 维度4: 忠实性 (actual 数字参数是否忠于 context)
        raise NotImplementedError

    def is_successful(self):
        # TODO: 你的代码
        raise NotImplementedError

metric = MarketingQualityMetric()
metric.measure(test_cases[0])
print(f"总分: {metric.score:.2f}, 通过: {metric.is_successful()}")


## TODO 3：批量评估营销文案集，输出评分矩阵

用自定义 metric 对所有 LLMTestCase 批量评估，输出每条文案的四维度评分和总分。

**任务**：写 `evaluate_all(test_cases, metric)` 函数，返回评分矩阵（list of dict）。

> 注：deepeval 也提供 `from deepeval import evaluate` 的批量 API，但需要配置评估模型。本任务用手动循环 `metric.measure(tc)` 实现规则评估（无 API），效果等价。

In [ ]:
# ===== 你的代码 =====
def evaluate_all(test_cases, metric):
    # TODO: 你的代码 —— 循环 measure，返回评分矩阵
    raise NotImplementedError

results = evaluate_all(test_cases, MarketingQualityMetric())
print(f"评估完成: {len(results)} 条")


## TODO 4：用 langsmith @traceable 追踪部署后营销 LLM 调用

模拟部署后的营销文案生成 LLM（mock，无真实 API），用 `@traceable` 装饰器追踪调用链：
- 记录输入（产品信息）
- 记录输出（生成文案）
- 记录延迟（模拟推理时间）

**任务**：用 `@traceable` 装饰 `marketing_llm_generate` 函数，运行 3 次调用并打印 trace 信息。

> 注：无 `LANGSMITH_API_KEY` 时 `@traceable` 仍可运行（本地模式，trace 存内存）。

In [ ]:
# ===== 你的代码 =====
# 用 @traceable 装饰 marketing_llm_generate，追踪调用
@traceable(name="marketing_llm_generate")
def marketing_llm_generate(product_brief: str, style: str = "小红书") -> dict:
    # TODO: 你的代码 —— 模拟 LLM 推理 + 返回 trace 信息
    raise NotImplementedError

# 运行 3 次调用并打印 trace
raise NotImplementedError


## TODO 5：用 tiktoken 监控日均万次营销文案生成的 token 成本

营销 LLM 部署后日均 10000 次调用，需要监控推理成本：
- 用 tiktoken 精确统计 input/output token（gpt-4o 用 `o200k_base`，DeepSeek V3 用 `cl100k_base`）
- 结合模型定价计算日均/月均成本
- 对比 gpt-4o vs DeepSeek V3（MoE 架构，成本仅 1/10）

**任务**：实现 `estimate_cost(text_in, text_out, daily_calls, model)` 函数。

In [ ]:
# ===== 你的代码 =====
def estimate_cost(text_in, text_out, daily_calls, model):
    # TODO: 你的代码 —— 用 tiktoken 统计 token + 计算成本
    # gpt-4o: o200k_base, $2.50/M in, $10.00/M out
    # DeepSeek V3: cl100k_base, $0.27/M in, $1.10/M out
    raise NotImplementedError

sample_input = "为智能手表Pro写小红书种草文案，卖点：7天续航、100+运动模式、心率血氧监测。"
sample_output = "姐妹们！这款智能手表Pro真的绝了！续航7天不充电，100+运动模式全覆盖，心率血氧24小时守护。"
raise NotImplementedError


## TODO 6：用 LLM-as-a-Judge 规则近似实现自动评分

LLM-as-a-Judge 通常用强 LLM（如 GPT-4）评判弱 LLM 输出。无 API 时，用规则近似实现：
- 关键词匹配（卖点覆盖）
- 长度检测（过短=跑题）
- CTA 检测（有无行动号召）
- 违禁词检测（无害性）

**任务**：实现 `LLMJudge` 类，对每条文案自动评分并输出诊断报告。

In [ ]:
# ===== 你的代码 =====
class LLMJudge:
    def __init__(self):
        self.forbidden_words = ["北方人", "南方人", "歧视"]
        self.cta_signals = ["快冲", "必买", "推荐", "首选"]

    def judge(self, actual, expected, input_text):
        # TODO: 你的代码 —— 实现 5 维度规则评分
        # 维度: 卖点覆盖 / 长度适中 / 有CTA / 无违禁 / 无幻觉
        raise NotImplementedError

judge = LLMJudge()
# TODO: 你的代码 —— 对所有样本评分并输出报告
raise NotImplementedError


## 总结

### 你完成了什么

1. **构建营销领域评测集**：5 条真实文案样本，覆盖优质/错误/违规/幻觉/跑题五种失败模式
2. **自定义 deepeval BaseMetric**：四维度规则评分（准确性/相关性/无害性/忠实性），无 API 的 LLM-as-a-Judge fallback
3. **批量评估营销文案**：输出评分矩阵，识别质量瓶颈
4. **LangSmith 追踪部署后 LLM**：@traceable 记录调用链、延迟、token
5. **tiktoken 监控推理成本**：日均万次请求的 gpt-4o vs DeepSeek V3 成本对比
6. **LLM-as-a-Judge 规则近似**：自动评分 + 诊断报告（卖点/长度/CTA/违禁/幻觉）

### 关键收获

- **评估三层框架**：通用基准（MMLU/HumanEval/AgentBench）→ 任务评测集 → 系统效果 A/B
- **deepeval 是 LLM 评估的 pytest**：自定义 BaseMetric 可嵌入 CI/CD
- **LangSmith 是 LLM 应用的 APM**：没有追踪的 LLM 应用等于黑箱
- **推理成本是部署核心瓶颈**：DeepSeek V3（MoE）用 1/10 成本逼近 gpt-4o 质量

### 2026 前沿关键词

`deepeval` · `LLM-as-a-Judge` · `LangSmith` · `vLLM` · `投机解码` · `MoE` · `AgentBench` · `RAGAS`

### 下一步

- 用 deepeval 评估自己构建的营销 RAG 系统（Day 2 的 RAGAS 指标可复用）
- 设计 A/B 测试验证 LLM 优化的业务价值
- 阅读 vLLM / 投机解码论文，理解推理优化架构